# Day 13 · Exercise 3: Index a Corpus

**What you'll build:** a function that takes a list of documents, chunks each one, embeds each chunk with `nomic-embed-text`, and stores everything in a ChromaDB collection — the ingestion pipeline for RAG.

**Why it matters:** the index is what makes instant retrieval possible — without it, every query would have to scan every document from scratch.

## Your Implementation

In [ ]:
import chromadb
import ollama

# You can use chunk_document from Exercise 2 — paste it here or re-implement it:
def chunk_document(text: str, chunk_size: int = 400, overlap: int = 50) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be less than chunk_size")
    chunks = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += step
    return chunks


def index_corpus(
    docs: list[dict],
    collection,
    chunk_size: int = 400,
    overlap: int = 50,
) -> int:
    """Chunk, embed, and store a list of documents in a ChromaDB collection.

    Args:
        docs: List of dicts, each with keys 'text' (str) and 'source' (str).
        collection: A ChromaDB Collection to add chunks to.
        chunk_size: Characters per chunk (passed to chunk_document).
        overlap: Overlap characters between chunks (passed to chunk_document).

    Returns:
        Total number of chunks stored in the collection.

    Example:
        >>> client = chromadb.Client()
        >>> coll = client.get_or_create_collection("test")
        >>> docs = [{"text": "Hello world. " * 50, "source": "doc_a"}]
        >>> count = index_corpus(docs, coll)
        >>> isinstance(count, int) and count > 0
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work
Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score = 0
    total = 4

    # Setup: in-memory Chroma collection
    try:
        import chromadb as _chromadb
        _client = _chromadb.Client()
        _coll = _client.get_or_create_collection("ex03_test")
    except Exception as e:
        print(f"{_FAIL} Setup: could not create ChromaDB collection: {e}")
        return

    _docs = [
        {"text": "Python is a high-level programming language. " * 5, "source": "python.txt"},
        {"text": "Machine learning trains models on data. " * 5, "source": "ml.txt"},
    ]

    # Check 1: returns an int
    try:
        result = index_corpus(_docs, _coll)
        if not isinstance(result, int):
            print(f"{_FAIL} Check 1: index_corpus should return an int, got {type(result).__name__}")
            return
        print(f"{_PASS} Check 1: index_corpus returns an int")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1: raised {type(e).__name__}: {e}")
        return

    # Check 2: return value > 0
    if result > 0:
        print(f"{_PASS} Check 2: returned {result} chunks (> 0)")
        score += 1
    else:
        print(f"{_FAIL} Check 2: returned {result}, expected > 0 chunks")
        return

    # Check 3: collection has the right number of chunks
    try:
        stored = _coll.count()
        if stored == result:
            print(f"{_PASS} Check 3: collection.count() == returned value ({stored})")
            score += 1
        else:
            print(f"{_FAIL} Check 3: collection has {stored} items but index_corpus returned {result}")
    except Exception as e:
        print(f"{_FAIL} Check 3: could not verify collection count: {e}")

    # Check 4: chunks have correct metadata keys (source, chunk_index)
    try:
        items = _coll.get(limit=1)
        meta = items["metadatas"][0] if items.get("metadatas") else {}
        if "source" in meta and "chunk_index" in meta:
            print(f"{_PASS} Check 4: chunks have 'source' and 'chunk_index' metadata")
            score += 1
        else:
            print(f"{_FAIL} Check 4: expected metadata keys 'source' and 'chunk_index', got {list(meta.keys())}")
    except Exception as e:
        print(f"{_FAIL} Check 4: could not inspect metadata: {e}")

    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed.")

_run_checks()

## Bonus Challenge
Add a `upsert=True` parameter to `index_corpus`. When `True`, use `collection.upsert()` instead of `collection.add()`, so the function can be called multiple times without duplicating chunks. (Hint: `collection.upsert()` takes the same arguments as `collection.add()` — from Day 12.)

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def index_corpus(
    docs: list[dict],
    collection,
    chunk_size: int = 400,
    overlap: int = 50,
) -> int:
    total = 0
    for doc in docs:
        chunks = chunk_document(doc["text"], chunk_size, overlap)
        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc['source']}_{i:04d}"
            emb = ollama.embeddings(model="nomic-embed-text", prompt=chunk)
            collection.add(
                ids=[chunk_id],
                embeddings=[emb["embedding"]],
                documents=[chunk],
                metadatas=[{"source": doc["source"], "chunk_index": i}],
            )
            total += 1
    return total
```

**Why this works:** each chunk gets a deterministic ID (`source_0000`, `source_0001`, …) so you can identify where each chunk came from. Embedding at index time — once per chunk — is what makes query time fast: you only embed the short question, not the entire corpus, on every search.
</details>